In [3]:
ROOT = r"/Users/julianaileru/Library/CloudStorage/GoogleDrive-jaileru@researchallofus.org/My Drive/ST002118/July-2026"
import sys,os
sys.path.append(os.path.abspath(ROOT))

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from Analysis.ST002118 import *

## 1-Load Data
* Untargeted dataset (MW:ST002118) - ST002118/July-2026/Data/Untargeted/ST002118_Untargeted_Reference_Copy.csv
* Targeted dataset (HHEAR ID:2018-2254) - ST002118/July-2026/Data/Targeted/TARGETED_DATA_WIDE_07132026.xlsx (converted to wide format)
* Epidemiological dataset (HHEAR ID:2018-2254) - ST002118/July-2026/Data/EPI_DATA/2254_EPI_DATA.csv

In [6]:
untargeted_data = pd.read_csv(rf"{ROOT}/Data/Untargeted/ST002118_Untargeted_Reference_Copy.csv").set_index("SID_MW")
targeted_data = pd.read_excel(rf"{ROOT}/Data/Targeted/TARGETED_DATA_WIDE_07132026.xlsx")
targeted_data = targeted_data.set_index("SID_MW")
annotations = pd.read_excel(rf"{ROOT}/Data/Annotations/All ADAP IDs and Annotations Lichtveld 09JUL26.xlsx")
annotations = annotations.set_index("query_signal_name")
epi_data = pd.read_csv(rf"{ROOT}/Data/EPI_DATA/2254_EPI_DATA.csv")
covariates = {"MaternalAgeAtIntake":"v107","MaternalEducationLevel":"v112a","Ethnicity6Categories":"v113c","BMI":"v111c","HouseholdIncome8Categories":"v115a"}
covariates = {v:k for k,v in covariates.items()}
pid_map = untargeted_data['PID'].to_dict()
# Drop this sample (ethnicity category is unique to this sample)
print({x:y for x,y in pid_map.items() if y == 9818485})
outcomes = {"b031":"Birthweight(grams)",
            "b034b":"GestationalAgeAtBirth"}
epi_data = epi_data.rename(columns=covariates)
epi_data = epi_data.rename(columns=outcomes)

{'Lich_S_127': 9818485, 'Lich_S_383': 9818485}


In [7]:
targeted_data = targeted_data.drop(index=['Lich_S_127',"Lich_S_383"])
t1_samples = targeted_data[targeted_data['Trimester_r'] == "1st/2nd"].index
t1_pid = untargeted_data.loc[t1_samples,"PID"]
untargeted_data_t1 = untargeted_data.loc[t1_samples,:]
targeted_data_t1 = targeted_data.loc[t1_samples,:]
epi_data_t1 = epi_data[epi_data['PID'].isin(t1_pid)].copy()

In [8]:
t2_samples = targeted_data[targeted_data['Trimester_r'] == "3rd"].index
t2_pid = untargeted_data.loc[t2_samples,"PID"]
untargeted_data_t2 = untargeted_data.loc[t2_samples,:]
targeted_data_t2 = targeted_data.loc[t2_samples,:]
epi_data_t2 = epi_data[epi_data['PID'].isin(t2_pid)].copy()

In [9]:
education_map = {
    1: "Low",
    2: "Low",
    3: "Middle",
    4: "Middle",
    5: "High",
    6: "High",
    7: "High"
}

income_map = {
    1: "0–800",
    2: "0–800",
    3: "800–1500",
    4: "1500–3000",
    5: "3000–5000",
    6: "5000+",
    7: "5000+"
}

In [10]:
epi_data_t1 = epi_data_t1[["PID","MaternalAgeAtIntake",
                            "MaternalEducationLevel","Ethnicity6Categories",
                            "BMI","HouseholdIncome8Categories",'Birthweight(grams)',"GestationalAgeAtBirth"]]

epi_data_t2 = epi_data_t2[["PID","MaternalAgeAtIntake",
                            "MaternalEducationLevel","Ethnicity6Categories",
                            "BMI","HouseholdIncome8Categories",'Birthweight(grams)','GestationalAgeAtBirth']]

In [11]:
# Collapse Categories into bigger sizes
epi_data_t1['MaternalEducationLevel_r'] = epi_data_t1['MaternalEducationLevel'].map(education_map)
epi_data_t1['HouseholdIncome_r'] = epi_data_t1['HouseholdIncome8Categories'].map(income_map)
print(epi_data_t1['HouseholdIncome_r'].value_counts().sum())
print(epi_data_t1['MaternalEducationLevel_r'].value_counts().sum())
print(epi_data_t1['Ethnicity6Categories'].value_counts().sum())
print(epi_data_t1['BMI'].isna().sum())
print("--"*5)
epi_data_t2['MaternalEducationLevel_r'] = epi_data_t2['MaternalEducationLevel'].map(education_map)
epi_data_t2['HouseholdIncome_r'] = epi_data_t2['HouseholdIncome8Categories'].map(income_map)
print(epi_data_t2['HouseholdIncome_r'].value_counts().sum())
print(epi_data_t2['MaternalEducationLevel_r'].value_counts().sum())
print(epi_data_t2['Ethnicity6Categories'].value_counts().sum())
print(epi_data_t2['BMI'].isna().sum())


309
309
309
0
----------
245
245
245
0


## Aim 1: Metals-MWAS (Association between Metal Exposure and Metabolites)

In [47]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def _standardize(data, feature):
    """Log-transform and z-score a metabolite feature."""
    x = np.log1p(data[feature])

    sd = x.std(ddof=1)
    if np.isnan(sd) or sd == 0:
        raise ValueError(f"{feature} has zero variance after log transformation.")

    data[feature] = (x - x.mean()) / sd


def MWAS(data,exposure,covariates):
    data = data.copy()
    metals = {"Hg", "Pb", "Se", "Cd", "Mn", "Sn"}

    metabolite_features = [col for col in data.columns if col.endswith(("n", "m/z")) and col not in metals]
    valid_features = []

    for feature in metabolite_features:
        try:
            _standardize(data, feature)
            valid_features.append(feature)
        except ValueError as e:
            print(e)

    metal_term = f"np.log1p({exposure})"
    cov_terms = " + ".join(covariates)

    results_list = []

    for feature in valid_features:

        formula = (f"Q('{feature}') ~ {metal_term} + {cov_terms}")

        model = smf.ols(formula=formula, data=data).fit(cov_type='HC3')

        results = pd.DataFrame({
            "feature": feature,
            "term": model.params.index,
            "coef": model.params.values,
            "std_err": model.bse.values,
            "t_value": model.tvalues.values,
            "p_value": model.pvalues.values,
        })

        ci = model.conf_int()
        results["CI_lower"] = ci[0].values
        results["CI_upper"] = ci[1].values
        results = results[results["term"].str.contains(exposure)]
        results["metal"] = results["term"].str.extract(r"log1p\((.*?)\)")
        results = results[
            [
                "feature",
                "metal",
                "coef",
                "std_err",
                "t_value",
                "p_value",
                "CI_lower",
                "CI_upper",
            ]
        ]

        results_list.append(results)
    final = pd.concat(results_list, ignore_index=True)
    final = final.dropna(subset='p_value')
    final["FDR"] = multipletests(
        final["p_value"],
        method="fdr_bh"
    )[1]
    return final

In [34]:
t1 = untargeted_data_t1.copy()
t1 = pd.merge(t1,targeted_data_t1,right_index=True,left_index=True)
t1['PID'] = t1.index.map(pid_map)
t1 = pd.merge(t1,epi_data_t1,right_on='PID',left_on='PID')
t1.columns = t1.columns.str.replace("(ng/mL)","").str.replace("(ug/dL)","").str.strip()

t2 = untargeted_data_t2.copy()
t2 = pd.merge(t2,targeted_data_t2,right_index=True,left_index=True)
t2['PID'] = t2.index.map(pid_map)
t2 = pd.merge(t2,epi_data_t2,right_on='PID',left_on='PID')
t2.columns = t2.columns.str.replace("(ng/mL)","").str.replace("(ug/dL)","").str.strip()

In [48]:
metals = ["Hg", "Pb", "Se", "Cd", "Mn","Sn"]
covariates = ["MaternalAgeAtIntake","C(MaternalEducationLevel_r)","C(HouseholdIncome_r)","C(Ethnicity6Categories)","BMI"]
results = {}
for metal in metals:
    print(metal)
    result = MWAS(exposure=metal, data=t1,covariates=covariates)
    result = pd.merge(result,annotations,left_on='feature',right_index=True,how='outer')
    results[metal] = result

with pd.ExcelWriter("Metals-MWAS-T1-080526.xlsx") as writer:
    for metal in metals:
        results[metal].to_excel(writer,sheet_name=metal,index=False)

Hg
7.59_346.0799n has zero variance after log transformation.
Pb
7.59_346.0799n has zero variance after log transformation.
Se
7.59_346.0799n has zero variance after log transformation.
Cd
7.59_346.0799n has zero variance after log transformation.
Mn
7.59_346.0799n has zero variance after log transformation.
Sn
7.59_346.0799n has zero variance after log transformation.


In [49]:
results = {}
for metal in metals:
    print(metal)
    result = MWAS(exposure=metal, data=t2,covariates=covariates)
    result = pd.merge(result,annotations,left_on='feature',right_index=True,how='outer')
    results[metal] = result

with pd.ExcelWriter("Metals-MWAS-T2-080526.xlsx") as writer:
    for metal in metals:
        results[metal].to_excel(writer,sheet_name=metal,index=False)

Hg
Pb
Se
Cd
Mn
Sn


## Aim 2: Metals Pathway Analysis (Find Pathways Perturbed by Metabolites Associated with Metals)

In [50]:
xlsx = pd.ExcelFile(rf"{ROOT}/Analysis/Metals-MWAS/Metals-MWAS-T1-080526.xlsx")
for sheet_name in xlsx.sheet_names:
    df = xlsx.parse(sheet_name,header=0)
    df = df.dropna(subset='p_value')
    t_table = df.copy()
    mummichog = t_table[['feature','t_value','p_value']].copy()
    mummichog[['retention time','m/z']] = mummichog['feature'].str.split("_",expand=True)
    mummichog['m/z'] = mummichog['m/z'].str.replace('m/z',"")
    mummichog['m/z'] = mummichog['m/z'].str.replace("n","")
    mummichog['m/z'] = mummichog['m/z'].astype(float)
    mask = mummichog['feature'].str.contains("n")
    mummichog.loc[mask, 'm/z'] = mummichog.loc[mask, 'm/z'] + 1.0074
    col_map = {"t_value":"t.stat","p_value":"p.value","retention time":"r.t","m/z":"m.z"}
    mummichog = mummichog.rename(columns=col_map)
    mummichog = mummichog.loc[:,list(col_map.values())]
    mummichog.to_csv(f'{ROOT}/Analysis/PathwayAnalysis/input/080526/{sheet_name}_mummichog_input-T1-080526.txt',sep='\t',index=False)

xlsx = pd.ExcelFile(rf"{ROOT}/Analysis/Metals-MWAS/Metals-MWAS-T2-080526.xlsx")
for sheet_name in xlsx.sheet_names:
    df = xlsx.parse(sheet_name,header=0)
    df = df.dropna(subset='p_value')
    t_table = df.copy()
    mummichog = t_table[['feature','t_value','p_value']].copy()
    mummichog[['retention time','m/z']] = mummichog['feature'].str.split("_",expand=True)
    mummichog['m/z'] = mummichog['m/z'].str.replace('m/z',"")
    mummichog['m/z'] = mummichog['m/z'].str.replace("n","")
    mummichog['m/z'] = mummichog['m/z'].astype(float)
    mask = mummichog['feature'].str.contains("n")
    mummichog.loc[mask, 'm/z'] = mummichog.loc[mask, 'm/z'] + 1.0074
    col_map = {"t_value":"t.stat","p_value":"p.value","retention time":"r.t","m/z":"m.z"}
    mummichog = mummichog.rename(columns=col_map)
    mummichog = mummichog.loc[:,list(col_map.values())]
    mummichog.to_csv(f'{ROOT}/Analysis/PathwayAnalysis/input/080526/{sheet_name}_mummichog_input-T2-080526.txt',sep='\t',index=False)

In [65]:
# Run-Pathway-Analysis (Rscript)
# p-value = 0.05
# rt = 3.0 (min)
# ion_mode = positive
# output_dir = ST002118/Analysis/PathwayAnalysis/output
!Rscript "../PathwayAnalysis/run_pathway_analysis.R"

[1] "MetaboAnalyst R objects initialized ..."
trying URL 'https://www.metaboanalyst.ca/resources/libs/mummichog/hsa_kegg.qs'
Content type 'unknown' length 358539 bytes (350 KB)
downloaded 350 KB

[1] "Got 8161 mass features."
[1] "2717 initial ECs created!"
Loading required package: plyr

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union

[1] "556 empirical compounds identified in 0.82026195526123 seconds."
[1] "A total of 1754 of duplicates were merged."
[1] "A total of 1768 of duplicates were merged."
[1] "A total of 1768 of duplicates were merged."
[1] "A total of 1754 of duplicates were merged."
[1] "A total of 1448 of duplicates were merged."
[1] "A total of 1448 of duplicates were merged."
[1] "A total of 1754 of duplicates were merged."
[1] "A total of 1768 of duplicates were merged."
[1] "Resampling,  100 permutations to estimate background ..."
[1] "Comple

In [68]:
import glob

list_ = glob.glob(r"../PathwayAnalysis/output/080526/*-T1-080526_pathway_enrichment_mummichog.csv")
pathways_t1 = {}
for file in list_:
    df = pd.read_csv(file)
    df = df.rename(columns={"Unnamed: 0":"Pathway"})
    df['Name'] = df['Pathway Number'] + "." + df['Pathway']
    df = df.sort_values(by='P(Fisher)',ascending=True)
    pw = os.path.basename(file).split("_")[0]
    pathways_t1[pw] = df[df['P(Fisher)'] < 0.1]['Pathway'].to_list()
    name = df['Name'].tolist()[:11]
    p_value = df['P(Fisher)'].tolist()[:11]
    print(os.path.basename(file))
    print("---"*5)
    j = list(zip(name,p_value))
    print("\n".join(f"{n}: {round(p,4)}" for n, p in j))
    print("-"*5)

print("---"*20)
list_ = glob.glob(r"../PathwayAnalysis/output/080526/*-T2-080526_pathway_enrichment_mummichog.csv")
pathways_t2 = {}
for file in list_:
    df = pd.read_csv(file)
    df = df.rename(columns={"Unnamed: 0":"Pathway"})
    df['Name'] = df['Pathway Number'] + "." + df['Pathway']
    df = df.sort_values(by='P(Fisher)',ascending=True)
    pw = os.path.basename(file).split("_")[0]
    pathways_t2[pw] = df[df['P(Fisher)'] < 0.1]['Pathway'].to_list()
    name = df['Name'].tolist()[:11]
    p_value = df['P(Fisher)'].tolist()[:11]
    print(os.path.basename(file))
    print("---"*5)
    j = list(zip(name,p_value))
    print("\n".join(f"{n}: {round(p,4)}" for n, p in j))
    print("-"*5)

Mn_mummichog_output-T1-080526_pathway_enrichment_mummichog.csv
---------------
P36.Steroid hormone biosynthesis: 0.0001
P1.Primary bile acid biosynthesis: 0.0043
P2.Steroid biosynthesis: 0.0867
P3.Butanoate metabolism: 0.2416
P4.Lysine degradation: 0.2512
P5.Histidine metabolism: 0.2769
P6.Nitrogen metabolism: 0.3259
P7.Porphyrin metabolism: 0.3536
P8.Drug metabolism - other enzymes: 0.4093
P9.Arginine and proline metabolism: 0.4297
P10.Pyrimidine metabolism: 0.4551
-----
Hg_mummichog_output-T1-080526_pathway_enrichment_mummichog.csv
---------------
P1.Linoleic acid metabolism: 0.0139
P2.Pentose and glucuronate interconversions: 0.0432
P3.Fructose and mannose metabolism: 0.0472
P4.Glycerophospholipid metabolism: 0.0641
P5.Lysine degradation: 0.1124
P6.Butanoate metabolism: 0.1329
P7.Nitrogen metabolism: 0.1493
P8.Biotin metabolism: 0.1593
P9.Metabolism of xenobiotics by cytochrome P450: 0.1893
P10.Pyrimidine metabolism: 0.1926
P11.Valine, leucine and isoleucine degradation: 0.2143
----

## Aim 3: Outcome Model (Association between Outcomes and Metabolites)
- Outcomes: GestationalAgeAtBirth, Birthweight

In [59]:
def outcome_model(data, metal_term, outcome, 
covariates=["MaternalAgeAtIntake",'C(MaternalEducationLevel_r)','C(HouseholdIncome_r)',"C(Ethnicity6Categories)","BMI"], feature_suffixes=("n", "m/z")):
    cov_terms = "+".join(covariates)
    final = []
    for feature in data.columns:
        if feature.endswith(feature_suffixes):
            formula = f"Q('{outcome}') ~ np.log1p(Q('{feature}')) + {cov_terms}"
            model = smf.ols(formula, data=data).fit()
            results = pd.DataFrame({
                "feature": feature,
                "term": model.params.index,
                "coef": model.params.values,
                "std_err": model.bse.values,
                "t_value": model.tvalues.values,
                "p_value": model.pvalues.values,
            })
            ci = model.conf_int()
            results["CI_lower"] = ci[0].values
            results["CI_upper"] = ci[1].values
            results = results[results["term"] == f"np.log1p(Q('{feature}'))"]
            results = results[["feature", "coef", "std_err", "t_value", "p_value", "CI_lower", "CI_upper"]]
            final.append(results)

    final = pd.concat(final)
    final = final.dropna(subset='p_value')
    final["FDR"] = multipletests(final["p_value"].astype(float), method='fdr_bh')[1]
    return final

In [60]:
metals_mwas_t1 = pd.ExcelFile(r"Metals-MWAS-T1-080526.xlsx")
Hg_mwas_t1 = metals_mwas_t1.parse(sheet_name='Hg')
Pb_mwas_t1 = metals_mwas_t1.parse(sheet_name='Pb')
Cd_mwas_t1 = metals_mwas_t1.parse(sheet_name='Cd')
Se_mwas_t1 = metals_mwas_t1.parse(sheet_name='Se')
Mn_mwas_t1 = metals_mwas_t1.parse(sheet_name='Mn')
Sn_mwas_t1 = metals_mwas_t1.parse(sheet_name='Sn')

metals_mwas_t2 = pd.ExcelFile(r"Metals-MWAS-T2-080526.xlsx")
Hg_mwas_t2 = metals_mwas_t2.parse(sheet_name='Hg')
Pb_mwas_t2 = metals_mwas_t2.parse(sheet_name='Pb')
Cd_mwas_t2 = metals_mwas_t2.parse(sheet_name='Cd')
Se_mwas_t2 = metals_mwas_t2.parse(sheet_name='Se')
Mn_mwas_t2 = metals_mwas_t2.parse(sheet_name='Mn')
Sn_mwas_t2 = metals_mwas_t2.parse(sheet_name='Sn')

In [61]:
Birthweight_t1 = {}
GestationalAge_t1 = {}
for metal,data in [("Hg (ng/mL)",Hg_mwas_t1),("Pb (ug/dL)",Pb_mwas_t1),("Cd (ng/mL)",Cd_mwas_t1),("Se (ng/mL)",Se_mwas_t1),("Mn (ng/mL)",Mn_mwas_t1),("Sn (ng/mL)",Sn_mwas_t1)]:
    print(metal)
    candidate_features = data[data['FDR'] < 0.2]['feature'].to_list()
    print(len(candidate_features))
    if len(candidate_features) <= 0:
        continue
    unt = untargeted_data_t1.loc[:,candidate_features].copy()
    unt['PID'] = t1_pid.copy()
    unt['exposure'] = targeted_data_t1[metal]
    merge = pd.merge(unt,epi_data_t1,right_on='PID',left_on="PID")
    Birthweight_t1[metal] = outcome_model(merge,"exposure",outcome='Birthweight(grams)')
    GestationalAge_t1[metal] = outcome_model(merge,"exposure",outcome='GestationalAgeAtBirth')

print("--"*5)
Birthweight_t2 = {}
GestationalAge_t2 = {}
for metal,data in [("Hg (ng/mL)",Hg_mwas_t2),("Pb (ug/dL)",Pb_mwas_t2),("Cd (ng/mL)",Cd_mwas_t2),("Se (ng/mL)",Se_mwas_t2),("Mn (ng/mL)",Mn_mwas_t2),("Sn (ng/mL)",Sn_mwas_t2)]:
    print(metal)
    candidate_features = data[data['FDR'] < 0.2]['feature'].to_list()
    print(len(candidate_features))
    if len(candidate_features) <= 0:
        continue
    unt = untargeted_data_t2.loc[:,candidate_features].copy()
    unt['PID'] = t2_pid.copy()
    unt['exposure'] = targeted_data_t2[metal]
    merge = pd.merge(unt,epi_data_t2,right_on='PID',left_on="PID")
    Birthweight_t2[metal] = outcome_model(merge,"exposure",outcome='Birthweight(grams)')
    GestationalAge_t2[metal] = outcome_model(merge,"exposure",outcome='GestationalAgeAtBirth')

Hg (ng/mL)
1916
Pb (ug/dL)
1026
Cd (ng/mL)
0
Se (ng/mL)
66
Mn (ng/mL)
1
Sn (ng/mL)
142
----------
Hg (ng/mL)
85
Pb (ug/dL)
2
Cd (ng/mL)
1
Se (ng/mL)
706
Mn (ng/mL)
135
Sn (ng/mL)
0


In [64]:
for key,value in GestationalAge_t1.items():
    print(key)
    q = value[value['p_value'] < 0.05]
    print(q.shape)
print("---"*5)
for key,value in Birthweight_t1.items():
    print(key)
    q = value[value['p_value'] < 0.05]
    print(q.shape)
print("---"*5)
for key,value in GestationalAge_t2.items():
    print(key)
    q = value[value['p_value'] < 0.05]
    print(q.shape)
print("---"*5)
for key,value in Birthweight_t2.items():
    print(key)
    q = value[value['p_value'] < 0.05]
    print(q.shape)

Hg (ng/mL)
(262, 8)
Pb (ug/dL)
(101, 8)
Se (ng/mL)
(5, 8)
Mn (ng/mL)
(0, 8)
Sn (ng/mL)
(42, 8)
---------------
Hg (ng/mL)
(181, 8)
Pb (ug/dL)
(50, 8)
Se (ng/mL)
(4, 8)
Mn (ng/mL)
(0, 8)
Sn (ng/mL)
(18, 8)
---------------
Hg (ng/mL)
(1, 8)
Pb (ug/dL)
(0, 8)
Cd (ng/mL)
(0, 8)
Se (ng/mL)
(10, 8)
Mn (ng/mL)
(3, 8)
---------------
Hg (ng/mL)
(4, 8)
Pb (ug/dL)
(0, 8)
Cd (ng/mL)
(0, 8)
Se (ng/mL)
(22, 8)
Mn (ng/mL)
(4, 8)


In [62]:
with pd.ExcelWriter("GestationalAge-080526.xlsx") as writer:
    for key,value in GestationalAge_t1.items():
        value = pd.merge(value,annotations,left_on='feature',right_index=True,how='left')
        name = key.split(" ")[0]
        name += "-T1"
        value.to_excel(writer,sheet_name=name,index=False)
    for key,value in GestationalAge_t2.items():
        value = pd.merge(value,annotations,left_on='feature',right_index=True,how='left')
        name = key.split(" ")[0]
        name += "-T2"
        value.to_excel(writer,sheet_name=name,index=False)


In [63]:
with pd.ExcelWriter("Birthweight-080526.xlsx") as writer:
    for key,value in Birthweight_t1.items():
        value = pd.merge(value,annotations,left_on='feature',right_index=True,how='left')
        name = key.split(" ")[0]
        name += "-T1"
        value.to_excel(writer,sheet_name=name,index=False)
    for key,value in Birthweight_t2.items():
        value = pd.merge(value,annotations,left_on='feature',right_index=True,how='left')
        name = key.split(" ")[0]
        name += "-T2"
        value.to_excel(writer,sheet_name=name,index=False)